<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보충 코드, 저자: <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

- 다음 셀의 주석을 해제하고 실행하여 이 보너스 노트북에 필요한 추가 패키지 요구사항을 설치하세요:

In [ ]:
# pip install -r requirements-extra.txt

# 다양한 바이트 페어 인코딩(Byte Pair Encoding, BPE) 구현 비교

<br>
&nbsp;

## `tiktoken`의 BPE 사용하기

In [3]:
from importlib.metadata import version

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.9.0


In [4]:
import tiktoken

tik_tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, world. Is this-- a test?"

In [5]:
integers = tik_tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [6]:
strings = tik_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


In [7]:
print(tik_tokenizer.n_vocab)

50257


<br>
&nbsp;

## GPT-2에서 사용된 원본 BPE 구현 사용하기

In [8]:
from bpe_openai_gpt2 import get_encoder, download_vocab

In [9]:
download_vocab()

Fetching encoder.json: 1.04Mit [00:00, 3.69Mit/s]                                                   
Fetching vocab.bpe: 457kit [00:00, 2.53Mit/s]                                                       


In [10]:
orig_tokenizer = get_encoder(model_name="gpt2_model", models_dir=".")

In [11]:
integers = orig_tokenizer.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


In [12]:
strings = orig_tokenizer.decode(integers)

print(strings)

Hello, world. Is this-- a test?


<br>
&nbsp;

## Hugging Face transformers를 통한 BPE 사용하기

In [13]:
import transformers

transformers.__version__

/Users/sebastian/Developer/LLMs-from-scratch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'4.49.0'

In [14]:
from transformers import GPT2Tokenizer

hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [15]:
hf_tokenizer(strings)["input_ids"]

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

In [ ]:
from transformers import GPT2TokenizerFast

hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")

In [ ]:
hf_tokenizer_fast(strings)["input_ids"]

<br>
&nbsp;

## 처음부터 작성한 나만의 BPE 토크나이저 사용하기

In [16]:
import os
import sys
import io
import nbformat
import types

def import_from_notebook():
    def import_definitions_from_notebook(fullname, names):
        current_dir = os.getcwd()
        path = os.path.join(current_dir, "..", "05_bpe-from-scratch", fullname + ".ipynb")
        path = os.path.normpath(path)

        # 노트북 로드
        if not os.path.exists(path):
            raise FileNotFoundError(f"Notebook file not found at: {path}")

        with io.open(path, "r", encoding="utf-8") as f:
            nb = nbformat.read(f, as_version=4)

        # 가져온 함수와 클래스를 저장할 모듈 생성
        mod = types.ModuleType(fullname)
        sys.modules[fullname] = mod

        # 노트북 셀들을 탐색하고 함수나 클래스 정의만 실행
        for cell in nb.cells:
            if cell.cell_type == "code":
                cell_code = cell.source
                for name in names:
                    # 함수나 클래스 정의 확인
                    if f"def {name}" in cell_code or f"class {name}" in cell_code:
                        exec(cell_code, mod.__dict__)
        return mod

    fullname = "bpe-from-scratch"
    names = ["BPETokenizerSimple"]

    return import_definitions_from_notebook(fullname, names)

In [17]:
imported_module = import_from_notebook()
BPETokenizerSimple = getattr(imported_module, "BPETokenizerSimple", None)

tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=os.path.join("gpt2_model", "encoder.json"),
    bpe_merges_path=os.path.join("gpt2_model", "vocab.bpe")
)

In [18]:
integers = tokenizer_gpt2.encode(text)

print(integers)

[15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]


<br>
&nbsp;

## 빠른 성능 벤치마크

In [19]:
with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

### OpenAI 원본 GPT-2 토크나이저

In [20]:
%timeit orig_tokenizer.encode(raw_text)

3.84 ms ± 9.83 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


### Tiktoken OpenAI GPT-2 토크나이저

In [21]:
%timeit tik_tokenizer.encode(raw_text)

901 μs ± 6.27 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### Hugging Face OpenAI GPT-2 토크나이저

In [22]:
%timeit hf_tokenizer(raw_text)["input_ids"]

Token indices sequence length is longer than the specified maximum sequence length for this model (5145 > 1024). Running this sequence through the model will result in indexing errors


11 ms ± 94.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [23]:
%timeit hf_tokenizer(raw_text, max_length=5145, truncation=True)["input_ids"]

10.8 ms ± 180 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
%timeit hf_tokenizer_fast(raw_text)["input_ids"]

In [ ]:
%timeit hf_tokenizer_fast(raw_text, max_length=5145, truncation=True)["input_ids"]

### 교육 목적을 위한 나만의 GPT-2 토크나이저

In [24]:
%timeit tokenizer_gpt2.encode(raw_text)

9.37 ms ± 50.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
